# Train Shared Vision Backbone (Ball + Marker CNN)
This notebook clones the repository, extracts the merged `shared_vision` gold dataset from your Google Drive, and trains the ~70K-param Shared Encoder Backbone (ball regression head + marker segmentation head + marker heatmap head) defined in `train_cnn_2d_tracker_marker.py`.

In [ ]:
EVAL_DATASET_NAME = 'shared_vision'              # Dataset 8, 100% real -- used ONLY for evaluation below
TRAIN_DATASET_NAME = 'shared_vision_synthetic_mix'  # Dataset 9, 60% synthetic / 40% real -- used ONLY for training
VERSION = 'v2'  # v1 was trained on mislabeled ball positions (point-reflected on both axes --
                 # see docs/PROJECT_LOGBOOK.md, 2026-08-12). v2 has never actually been trained/
                 # uploaded yet, so it stays v2 here even though the data backing it now also
                 # includes Dataset 9 (60/40 synthetic/real marker mix, extends
                 # implementation_plan_shared_backbone_cnn.md's Component 3 with shape/color
                 # variation, not just position) -- see docs/PROJECT_LOGBOOK.md, 2026-08-12
                 # (Dataset 9 entry).

**Before running this for `v2`:** two separate datasets need to be zipped and uploaded to Drive -- training and evaluation deliberately read from different folders so evaluation never sees synthetic data:
1. `/content/drive/MyDrive/shared_vision.zip` -- the corrected, all-real Dataset 8 (`host_software/data/03_gold/shared_vision/` locally). Used only by the Evaluation section below. Re-upload if you haven't already since the 2026-08-12 ball-label fix.
2. `/content/drive/MyDrive/shared_vision_synthetic_mix.zip` -- Dataset 9, the 60% synthetic / 40% real training mix (`host_software/data/03_gold/shared_vision_synthetic_mix/` locally, built by `generate_synthetic_marker_composites.py` + `combine_shared_vision_training_mix.py`). Used only by the Training section below.

Zip and upload both before running the cells below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf /content/ball_balance_video_controlled
!git clone https://github.com/Jack0468/ball_balance_video_controlled.git
!pip install pandas torch torchvision albumentations opencv-python-headless matplotlib onnx onnxscript
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Unzip both 03_gold datasets (images/ + masks/ + labels.csv each) -- eval dataset
# (all-real) and train dataset (60/40 synthetic/real mix) are kept physically
# separate so evaluation never reads a synthetic row.
!mkdir -p /content/ball_balance_video_controlled/host_software/data/03_gold
!unzip -q -o /content/drive/MyDrive/{EVAL_DATASET_NAME}.zip -d /content/ball_balance_video_controlled/host_software/data/03_gold
!unzip -q -o /content/drive/MyDrive/{TRAIN_DATASET_NAME}.zip -d /content/ball_balance_video_controlled/host_software/data/03_gold
print("Datasets unzipped!")

### Training
`--resume` is always passed below: `output-dir` is on Google Drive, so if the Colab runtime disconnects or crashes mid-training, `shared_vision_backbone_resume.pt` (model + optimizer + scheduler state + loss history, saved after every epoch) survives there. Just re-run this cell -- it picks up from the last completed epoch instead of retraining from scratch. If no checkpoint exists yet (first run), `--resume` is a no-op and training starts fresh.

In [ ]:
%cd /content/ball_balance_video_controlled
# Run as a module (-m), not a plain script -- the trainer imports
# host_software.ml_vision.training.{shared_vision_dataset,augmentations} as package-qualified
# paths, which only resolve when the repo root is on sys.path (i.e. invoked via -m from here).
!python -m host_software.ml_vision.training.train_cnn_2d_tracker_marker \
    --csv-file host_software/data/03_gold/{TRAIN_DATASET_NAME}/labels.csv \
    --images-dir host_software/data/03_gold/{TRAIN_DATASET_NAME}/images \
    --mask-dir host_software/data/03_gold/{TRAIN_DATASET_NAME}/masks \
    --output-dir /content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION} \
    --resume

### Training Results
`train_cnn_2d_tracker_marker.py` exports the best checkpoint to ONNX and writes a per-session validation breakdown and loss curve.

In [ ]:
from IPython.display import Image, display
import pandas as pd

output_dir = f'/content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION}'
display(Image(filename=f'{output_dir}/training_curve.png'))
pd.read_csv(f'{output_dir}/per_session_eval.csv')

### Evaluation
Runs `evaluate_shared_vision_backbone.py` against `EVAL_DATASET_NAME` (Dataset 8, all-real) -- deliberately **not** the `TRAIN_DATASET_NAME` mix training just used, so reported metrics are never computed against synthetic frames. `evaluate_shared_vision_backbone.py` independently re-derives its own held-out temporal slice (same `--val-fraction` default) from that real-only CSV, disjoint from whatever training used, whether or not the two datasets overlap. Reports ball-position pixel error, marker mask IoU/Dice, heatmap MSE, and inference latency -- metrics `train_cnn_2d_tracker_marker.py` doesn't compute. Also saves an error-distribution histogram and a qualitative grid of predicted-vs-ground-truth ball points and mask contours.

In [ ]:
%cd /content/ball_balance_video_controlled
# Deliberately EVAL_DATASET_NAME (all-real, Dataset 8), not TRAIN_DATASET_NAME --
# reported metrics must never be computed against synthetic frames.
!python -m host_software.ml_vision.evaluations.evaluate_shared_vision_backbone \
    --csv-file host_software/data/03_gold/{EVAL_DATASET_NAME}/labels.csv \
    --images-dir host_software/data/03_gold/{EVAL_DATASET_NAME}/images \
    --mask-dir host_software/data/03_gold/{EVAL_DATASET_NAME}/masks \
    --checkpoint {output_dir}/shared_vision_backbone_best.pt

In [ ]:
import json

display(Image(filename=f'{output_dir}/evaluation_error_histogram.png'))
display(Image(filename=f'{output_dir}/evaluation_visual_grid.png'))
with open(f'{output_dir}/evaluation_metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))